# SteamDT 悠悠小时数据库 · VS Code 本地采集

版本：steamdt-research-1.0。运行前选择项目 Python 内核；只需 Python 3.11+、VS Code Jupyter 扩展，无需填写 Cookie 或登录凭证。

本入口读取 SteamDT 网页明确选择「悠悠」的小时 K 线，保留 OHLC 与来源量额，另行派生 4 小时、日和周数据。来源量额尚不能认定为 UU 真实成交记录。当前只采集、检查数据，不训练、不输出买卖建议。

**默认不会发起市场请求。** 先运行配置和清单单元格，核对商品；需要采集时再把 `RUN_COLLECTION` 改为 `True`，单独运行采集单元格。每轮有上限，中断后下次用同一工作区继续。


In [ ]:
from pathlib import Path
import sys, json
from IPython.display import display, Markdown

# 从项目根目录、scripts/notebooks 或它们的祖先定位项目。
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p for p in candidates if (p / "scripts/steamdt_research.py").is_file()), None)
if PROJECT is None:
    raise RuntimeError("请先在 VS Code 打开 F:/Codex/CS_YouOweMe 文件夹")
sys.path.insert(0, str(PROJECT))
from scripts import steamdt_research as engine

WORKSPACE = "research_20261016_v1"  # 日后继续用同名工作区，保留数据库与进度。
warehouse = engine.directory(WORKSPACE)
if not warehouse.exists():
    engine.init_workspace(WORKSPACE)  # 仅本地初始化，已有工作区不会覆盖。
engine.check_workspace(WORKSPACE)
print("商品清单：", warehouse / "registry.json")
print("数据库：", warehouse / "research.sqlite")
print("历史报告：", warehouse / "reports")


## 商品清单：以后只需维护这一个文件

在 VS Code 打开上方输出的 `registry.json`。每个条目代表「精确名称＋磨损＋普通版本」；增加商品时复制一个条目并修改字段，注意 JSON 逗号。

- `market_hash_name`：准确英文规格名，程序用它校验商品身份。
- `name_zh`：供你阅读的中文名。
- `steamdt_item_id`、`uu_template_id`：数字必须加双引号；不知道可填 `null`，程序尝试按英文规格解析，并输出 `resolved_items.csv`。已填编号若与来源不符会停止。
- `history_start`：默认 `"2025-06-01"`；填 `null` 表示尝试回溯至来源实际可提供的最早记录，可能需要多轮，不保证网站有全部历史。
- `enabled`：`true` 采集，`false` 暂停；暂停不会删除旧数据。

新商品自动建回补任务；旧商品续接未完成分页，并补最新区间。每轮开始重新读取清单，不必改采集代码。公开的 scripts/config 清单只是新工作区的初始模板，之后维护当前工作区清单即可。


In [ ]:
items = engine.registry(warehouse / "registry.json")
for item in items["items"]:
    print(f"{'启用' if item['enabled'] else '暂停'} | {item['name_zh']} | SteamDT={item['steamdt_item_id']} | UU={item['uu_template_id']} | 起点={item['history_start']}")


## 启动一轮有限采集

`daily` 补新增数据并重查最近 7 天，`weekly` 重查最近 35 天。重查用于发现来源修订，数据库不会因相同记录增加重复主数据；变化值与压缩原始响应保留证据。

默认请求间隔 5 秒、每轮最多 80 次（包括身份核验与时钟请求），单轮时间上限 15 分钟、共享滚动 24 小时最多 300 次。多商品和久远历史需要多轮；每轮仍受预算限制。完成或预算暂停后，如还需回补，手动再次运行本单元格。

可点 VS Code 的「中断」；程序保留已提交分页和游标。不要同时开两个采集进程。401/403/429、网络或结构异常会停止，不自动重试；正常访问恢复后按使用说明处理，不能跳过服务器冷却。


In [ ]:
RUN_COLLECTION = False  # 确认要采集时改 True；默认运行全部单元格不会联网。
MODE = "daily"          # daily 或 weekly
MAX_REQUESTS = 80       # 1..120，预算用完后下次继续
DELAY_SECONDS = 5       # 3..60，推荐保持 5 或更高
NORMAL_ACCESS_CONFIRMED = False  # 仅先前受限且已确认正常访问恢复时使用

if RUN_COLLECTION:
    result = engine.sync(WORKSPACE, mode=MODE, maximum=MAX_REQUESTS,
                         delay=DELAY_SECONDS, confirmed=NORMAL_ACCESS_CONFIRMED)
    print(json.dumps(result["run"], ensure_ascii=False, indent=2))
else:
    print("尚未启动采集。核对清单后，把 RUN_COLLECTION 改为 True，再运行本单元格。")


## 查看落盘结果

单元格只显示摘要；真正数据保存在本地 `research.sqlite`、`raw/`、`runs/`、`reports/`。下面读取最近一次完成落盘的报告，不会联网，也不修改数据库。采集中途的实时游标在 SQLite `jobs` 表；报告在本轮结束或正常中断后更新。

`paused_budget` 表示保存进度、等待下一轮；`completed` 表示当前任务已处理，不代表请求起点有完整历史。必须结合首末时间、区间缺口、任务 `finish_reason` 看覆盖。量额空值保留，不补零；当前尚未结束的小时不进入导出。


In [ ]:
saved = engine.status(WORKSPACE)
if "items" not in saved:
    print("工作区已初始化，尚无采集报告。")
else:
    print("最近一轮：", saved["run"]["status"], "；数据库完整性：", saved["integrity"])
    for item in saved["items"]:
        print(f"{item['name_zh']} | {item['hour_rows']}小时 | {item['first_hour_beijing']} → {item['last_hour_beijing']} | 区间缺{item['missing_between_first_last']}小时")
        print("  SteamDT:", item['steamdt_item_id'], "UU:", item['uu_template_id'],
              "起点有记录:", item['requested_start_observed'])
        for job in item['jobs']:
            print("  任务", job['job_id'], job['state'], "页数", job['pages'], "游标", job['cursor'], "结束原因", job['finish_reason'])
    pointer = json.loads((warehouse / "latest_report.json").read_text(encoding="utf-8"))
    report_dir = (warehouse / pointer["report"]).parent
    print("报告目录：", report_dir)
    display(Markdown((report_dir / "report.md").read_text(encoding="utf-8")))


## 日后怎么运行

日常至少每日运行一轮 `daily`，每周再运行一轮 `weekly` 做更深复核。若只每周更新，日中决策会使用过期信息。真正下单前还需要核对 UU 当前报价、属性匹配和实际可卖状态；历史预测仍是后续阶段。

保留整个工作区和 research_control 控制目录；只保留 SQLite 会丢失原始响应与控制记录。不要为了重跑删除数据库。详细说明见 `docs/usage/STEAMDT_RESEARCH.md`。
